# Fase 3 · Núcleo algorítmico, eficiencia y POO
## Análisis de licitaciones públicas de ChileCompra · Enero 2025

**MCDI500 · Grupo 2**

Partimos de `F2_Preprocesamiento`. Esa fase exploró la base y aplicó tres transformaciones al conjunto exportado: convirtió fechas, convirtió montos y ordenó el tamaño del proveedor. En esta fase las ejecutamos desde un módulo de Python, medimos dos formas de hacer un control temporal y verificamos que el resultado completo coincida con el de Fase 2.

**Para ejecutar:** mantén este notebook en `F3/`, el módulo en `src/` y el CSV original en `data/raw/04LicitacionesEne.csv`. Abre el notebook en el entorno del repositorio y ejecuta todas las celdas. Se requiere Python con pandas, scikit-learn y Jupyter.

## 0. Verificar el entorno de ejecución

Antes de cargar datos comprobamos que el notebook se ejecuta con el entorno virtual del proyecto (`.venv`, kernel **Python (grupo2-mcdi500)**) y que las librerías que usa esta fase tienen las versiones declaradas en `requirements.txt`. Es la misma verificación de la sección 8 de `F1_Definicion`, reducida a lo que necesita la Fase 3.

Importa porque las mediciones de tiempo y memoria de las secciones siguientes dependen del entorno: si cambia la versión de pandas, pueden cambiar las cifras. La celda se detiene si no hay entorno virtual o si falta una librería, y avisa si alguna versión no coincide con la declarada.

In [1]:
# Verificar el entorno: intérprete, entorno virtual y versiones frente a requirements.txt.
import platform
import sys
from importlib import metadata
from pathlib import Path

import pandas as pd
from IPython.display import display

# Paquetes que usa la Fase 3, con el nombre que tienen en requirements.txt.
PAQUETES_F3 = ["pandas", "numpy", "scikit-learn"]


def leer_versiones_declaradas(ruta_requirements, paquetes):
    """Devuelve {paquete: versión} de requirements.txt para los paquetes pedidos."""
    declaradas = {}
    for linea in ruta_requirements.read_text(encoding="utf-8").splitlines():
        nombre, _, version = linea.strip().partition("==")
        if nombre.lower() in paquetes:
            declaradas[nombre.lower()] = version
    return declaradas


def verificar_entorno(paquetes):
    """Comprueba el entorno virtual y compara versiones instaladas y declaradas.

    Se detiene si no hay entorno virtual o si falta un paquete. Devuelve una
    tabla con la versión declarada e instalada de cada paquete.
    """
    inicio = Path.cwd().resolve()
    raiz = next((p for p in (inicio, *inicio.parents) if (p / "requirements.txt").is_file()), None)
    if raiz is None:
        raise FileNotFoundError("No se encontró requirements.txt: abre el notebook dentro del repositorio.")
    if sys.prefix == sys.base_prefix:
        raise RuntimeError("No se usa un entorno virtual: cambia el kernel a 'Python (grupo2-mcdi500)'.")

    declaradas = leer_versiones_declaradas(raiz / "requirements.txt", paquetes)
    filas = []
    for paquete in paquetes:
        try:
            instalada = metadata.version(paquete)
        except metadata.PackageNotFoundError:
            instalada = None
        filas.append({"Paquete": paquete, "Declarada": declaradas.get(paquete),
                      "Instalada": instalada, "Coincide": instalada == declaradas.get(paquete)})
    tabla = pd.DataFrame(filas)

    faltantes = tabla.loc[tabla["Instalada"].isna(), "Paquete"].tolist()
    if faltantes:
        raise ModuleNotFoundError(f"Faltan paquetes: {faltantes}. Ejecuta: pip install -r requirements.txt")

    interprete = Path(sys.executable)
    if interprete.is_relative_to(raiz):
        interprete = interprete.relative_to(raiz)
    print(f"Intérprete: {interprete} | Python {platform.python_version()} | {platform.system()}")
    print("[OK] Entorno virtual activo")
    return tabla


versiones = verificar_entorno(PAQUETES_F3)
display(versiones)
if not versiones["Coincide"].all():
    print("[AVISO] Hay versiones distintas a las declaradas: las cifras de tiempo pueden variar.")
else:
    print(f"[OK] Las {len(versiones)} versiones coinciden con requirements.txt")

Intérprete: .venv/bin/python | Python 3.12.3 | Darwin
[OK] Entorno virtual activo


,Paquete,Declarada,Instalada,Coincide
0,pandas,3.1.0,3.1.0,True
1,numpy,2.5.4,2.5.4,True
2,scikit-learn,1.9.1,1.9.1,True


[OK] Las 3 versiones coinciden con requirements.txt


## 1. Cargar el mismo archivo que en Fase 2

Buscamos la raíz del repositorio a partir de la carpeta donde se ejecuta el notebook. Leemos el CSV con la misma codificación y detección de separador que usó Fase 2. Mostramos dimensiones y monedas para comprobar que la entrada es la esperada. La columna de moneda se conserva, por lo que los montos siguen siendo comparables dentro de su propia unidad.

In [2]:
from pathlib import Path
import sys
import platform
from timeit import repeat

import pandas as pd
import sklearn
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit

relativa = Path("data/raw/04LicitacionesEne.csv")
inicio = Path.cwd()
raiz = next((p for p in (inicio, *inicio.parents) if (p / relativa).is_file()), None)
if raiz is None:
    raise FileNotFoundError(f"Busca el archivo {relativa} dentro del repositorio.")
sys.path.insert(0, str(raiz))

from src.pipeline_licitaciones import (
    Bitacora, ConversorFechas, ConversorMontos, OrdenadorTamano, Pipeline,
    FECHAS, MONTOS, ORDEN, TAMANO, cargar_licitaciones,
)

datos_originales = cargar_licitaciones(raiz / relativa)
print(f"Python {platform.python_version()} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")
print(f"Entrada: {datos_originales.shape[0]:,} filas y {datos_originales.shape[1]} columnas")
display(datos_originales["MonedaOferta"].value_counts(dropna=False).to_frame("Filas"))

Python 3.12.3 | pandas 3.1.0 | scikit-learn 1.9.1
Entrada: 45,741 filas y 74 columnas


,Filas
MonedaOferta,
CLP,45604
CLF,105
USD,23
UTM,9


## 2. Ejecutar las transformaciones como objetos

| Clase | Qué hace | Regla conservada de Fase 2 |
|---|---|---|
| `ConversorFechas` | Convierte publicación, cierre y adjudicación | Formato día-mes-año y hora:minuto. Un valor inválido queda como `NaT`. |
| `ConversorMontos` | Convierte monto neto y total a números | Quita espacios exteriores y cambia coma decimal por punto. Los valores no interpretables quedan nulos. |
| `OrdenadorTamano` | Ordena el tamaño de proveedor | Usa Micro, Pequeña, Mediana, Grande y NoClasificado. Si aparece otra categoría, conserva el tipo original. |

**Encapsulamiento.** Cada conversor guarda su configuración internamente, por ejemplo `_formato` y `_orden`. El método público `transformar()` comprueba que existan las columnas antes de aplicar la regla.

**Herencia.** Las tres clases heredan de `Transformador` y reutilizan la comprobación de columnas. Cada hija define su propia operación en `_aplicar()`.

**Polimorfismo.** `Pipeline` llama a `paso.transformar(datos)` para cada objeto, con independencia del tipo de transformación. Las funciones de carga y medición quedan separadas de las clases.

**Patrón Observer.** Suscribimos la bitácora para que reciba los controles de cada paso. El pipeline puede ejecutar las mismas transformaciones con o sin esa bitácora.

**Decisión sobre recursividad.** El CSV es tabular y las transformaciones siguen una secuencia fija. Usamos división funcional y un bucle porque la cantidad de pasos es conocida y no hay una jerarquía de profundidad variable que recorrer.

In [3]:
bitacora = Bitacora()
pipeline = Pipeline([ConversorFechas(), ConversorMontos(), OrdenadorTamano()])
pipeline.suscribir(bitacora.registrar)

resultado_f3 = pipeline.transformar(datos_originales)
display(bitacora.tabla())
display(resultado_f3[[*FECHAS, *MONTOS, TAMANO]].dtypes.to_frame("Tipo final"))

,Paso,Filas entrada,Filas salida,Columna,Nulos originales,Fallos de conversión,Detalle
0,ConversorFechas,45741,45741,FechaPublicacion,0.0,0.0,NaN
1,ConversorFechas,45741,45741,FechaCierre,0.0,0.0,NaN
2,ConversorFechas,45741,45741,FechaAdjudicacion,91.0,0.0,NaN
3,ConversorMontos,45741,45741,MontoNetoOferta,0.0,0.0,NaN
4,ConversorMontos,45741,45741,MontoTotalOferta,0.0,0.0,NaN
5,OrdenadorTamano,45741,45741,TamanoProveedor,NaN,NaN,Categoría ordenada


,Tipo final
FechaPublicacion,datetime64[us]
FechaCierre,datetime64[us]
FechaAdjudicacion,datetime64[us]
MontoNetoOferta,Float64
MontoTotalOferta,Float64
TamanoProveedor,category


## 3. Comprobar la equivalencia con Fase 2

Reconstruimos la salida de Fase 2 con las instrucciones de conversión del notebook original. La referencia se calcula desde el CSV original en memoria y no llama al módulo de Fase 3. Comparamos los DataFrames completos, incluidos orden, tipos, categorías, valores y nulos. Así evitamos que la lectura de un CSV exportado cambie los tipos antes de comparar.

In [4]:
def ejecutar_referencia_f2(df):
    # Conversión de fechas: mismas reglas del paso 14 de Fase 2.
    df_trabajo = df.copy()
    for columna in ["FechaPublicacion", "FechaCierre", "FechaAdjudicacion"]:
        df_trabajo[columna] = pd.to_datetime(
            df[columna], format="%d-%m-%Y %H:%M", errors="coerce"
        )

    # Conversión de montos: misma regla del paso 18 de Fase 2.
    for columna in ["MontoNetoOferta", "MontoTotalOferta"]:
        original = df_trabajo[columna].astype("string").str.strip()
        texto_convertido = original.str.replace(",", ".", regex=False)
        df_trabajo[columna] = pd.to_numeric(texto_convertido, errors="coerce")

    # Categoría ordenada: misma condición de los pasos 26 y 27 de Fase 2.
    orden_tamano = ["Micro", "Pequeña", "Mediana", "Grande", "NoClasificado"]
    valores_actuales = set(df_trabajo["TamanoProveedor"].dropna().unique())
    if not valores_actuales - set(orden_tamano):
        tipo_tamano = pd.CategoricalDtype(categories=orden_tamano, ordered=True)
        df_trabajo["TamanoProveedor"] = df_trabajo["TamanoProveedor"].astype(tipo_tamano)
    return df_trabajo


resultado_f2 = ejecutar_referencia_f2(datos_originales)
pd.testing.assert_frame_equal(resultado_f2, resultado_f3, check_exact=True)
print(f"Equivalencia comprobada: {len(resultado_f3):,} filas y {resultado_f3.shape[1]} columnas.")

Equivalencia comprobada: 45,741 filas y 74 columnas.


## 4. Medir dos implementaciones del control temporal

Buscamos registros cuyo cierre antecede a la publicación o cuya adjudicación antecede al cierre. Una comparación se evalúa solo cuando sus dos fechas están presentes. Primero comprobamos las dos versiones con casos controlados y luego exigimos que marquen las mismas filas en el conjunto real.

Medimos solo el cálculo de la máscara, sobre todas las filas y con siete repeticiones por versión. Reportamos el menor tiempo observado. Ambas implementaciones recorren las filas y tienen costo temporal **O(n)**. La diferencia medida muestra cuánto cuesta ejecutar las comparaciones en Python frente a realizarlas por columnas con pandas. El tiempo depende del equipo. El control solo reporta señales y no cambia el conjunto.

In [5]:
def control_iterativo(datos):
    marcas = []
    for publicacion, cierre, adjudicacion in datos[list(FECHAS)].itertuples(index=False, name=None):
        cierre_anticipado = pd.notna(publicacion) and pd.notna(cierre) and cierre < publicacion
        adjudicacion_anticipada = pd.notna(cierre) and pd.notna(adjudicacion) and adjudicacion < cierre
        marcas.append(cierre_anticipado or adjudicacion_anticipada)
    return pd.Series(marcas, index=datos.index, dtype=bool)


def control_vectorizado(datos):
    publicacion, cierre, adjudicacion = (datos[c] for c in FECHAS)
    return (cierre < publicacion) | (adjudicacion < cierre)


casos = pd.DataFrame([
    ["2025-01-01", "2025-01-03", "2025-01-05"],
    ["2025-01-03", "2025-01-02", None],
    [None, "2025-01-05", "2025-01-04"],
    ["2025-01-01", None, "2025-01-05"],
], columns=FECHAS).apply(pd.to_datetime)
esperado = pd.Series([False, True, True, False])
pd.testing.assert_series_equal(control_iterativo(casos), esperado)
pd.testing.assert_series_equal(control_vectorizado(casos), esperado)

mascara = control_vectorizado(resultado_f3)
pd.testing.assert_series_equal(control_iterativo(resultado_f3), mascara)

mediciones = []
for nombre, operacion in [("Iterativa", control_iterativo), ("Vectorizada", control_vectorizado)]:
    tiempos = repeat(lambda: operacion(resultado_f3), number=1, repeat=7)
    mediciones.append({"Implementación": nombre, "Filas": len(resultado_f3),
                       "Repeticiones": len(tiempos), "Mínimo (ms)": min(tiempos) * 1000})

tabla_tiempos = pd.DataFrame(mediciones)
display(tabla_tiempos.round(3))
razon = tabla_tiempos.loc[0, "Mínimo (ms)"] / tabla_tiempos.loc[1, "Mínimo (ms)"]
print(f"Razón de tiempos iterativa/vectorizada: {razon:.2f}×")
print(f"Filas con alerta temporal: {mascara.sum():,}. Las dos versiones identificaron las mismas filas.")

,Implementación,Filas,Repeticiones,Mínimo (ms)
0,Iterativa,45741,7,80.534
1,Vectorizada,45741,7,0.145


Razón de tiempos iterativa/vectorizada: 555.25×
Filas con alerta temporal: 0. Las dos versiones identificaron las mismas filas.


## 5. Preparar los archivos para Fase 4

La verificación anterior usa todo el conjunto. Ahora creamos entrenamiento y prueba con una semilla fija y una proporción objetivo de 80/20 **por licitación**, de modo que las ofertas de una misma licitación queden juntas. El porcentaje por filas puede diferir. Como las conversiones usan reglas fijas, ningún paso estima medianas, escalas u otros parámetros a partir de los datos de prueba.

El diccionario documenta columnas, tipos y origen. El archivo de parámetros registra las reglas fijas y la semilla. Los CSV se guardan en `data/processed/` con punto y coma y UTF-8 con BOM. Al volver a leerlos, usa el diccionario para reconstruir fechas y categorías cuando haga falta.

In [6]:
semilla = 42
fraccion_prueba = 0.20
grupos = resultado_f3["NroLicitacion"]
if grupos.isna().any():
    raise ValueError("Hay licitaciones sin identificador. Revísalas antes de dividir por licitación.")

divisor = GroupShuffleSplit(n_splits=1, test_size=fraccion_prueba, random_state=semilla)
indices_ent, indices_pru = next(divisor.split(resultado_f3, groups=grupos))
entrenamiento = resultado_f3.iloc[indices_ent]
prueba = resultado_f3.iloc[indices_pru]

assert set(entrenamiento["NroLicitacion"]).isdisjoint(prueba["NroLicitacion"])
assert len(entrenamiento) + len(prueba) == len(resultado_f3)
pd.testing.assert_series_equal(entrenamiento.dtypes, prueba.dtypes)

reglas = {**dict.fromkeys(FECHAS, "Conversión de fecha y hora"),
          **dict.fromkeys(MONTOS, "Conversión numérica con coma decimal"),
          TAMANO: "Orden lógico de tamaño de proveedor"}
diccionario = pd.DataFrame([
    {"Columna": c, "Tipo": str(resultado_f3[c].dtype),
     "Origen": "04LicitacionesEne.csv", "Transformación": reglas.get(c, "Se conserva")}
    for c in resultado_f3.columns
])
parametros = pd.DataFrame([
    ("ConversorFechas", "formato", "%d-%m-%Y %H:%M", "Fijo"),
    ("ConversorMontos", "regla", "Recortar espacios y sustituir coma por punto", "Fijo"),
    ("OrdenadorTamano", "orden", " | ".join(ORDEN), "Fijo"),
    ("División", "grupo", "NroLicitacion", "Fijo"),
    ("División", "semilla", str(semilla), "Fijo"),
    ("División", "fracción de prueba por grupos", str(fraccion_prueba), "Fijo"),
], columns=["Paso", "Parámetro", "Valor", "Origen"])

destino = raiz / "data" / "processed"
destino.mkdir(parents=True, exist_ok=True)
archivos = {"entrenamiento_f3.csv": entrenamiento, "prueba_f3.csv": prueba,
            "diccionario_f3.csv": diccionario, "parametros_f3.csv": parametros}
for nombre, tabla in archivos.items():
    tabla.to_csv(destino / nombre, sep=";", encoding="utf-8-sig", index=False)
    print(f"data/processed/{nombre} · {len(tabla):,} filas")

print(f"Licitaciones únicas: {entrenamiento['NroLicitacion'].nunique():,} en entrenamiento y "
      f"{prueba['NroLicitacion'].nunique():,} en prueba. No hay grupos compartidos.")
display(parametros)

data/processed/entrenamiento_f3.csv · 36,438 filas
data/processed/prueba_f3.csv · 9,303 filas
data/processed/diccionario_f3.csv · 74 filas
data/processed/parametros_f3.csv · 6 filas
Licitaciones únicas: 1,833 en entrenamiento y 459 en prueba. No hay grupos compartidos.


,Paso,Parámetro,Valor,Origen
0,ConversorFechas,formato,%d-%m-%Y %H:%M,Fijo
1,ConversorMontos,regla,Recortar espacios y sustituir coma por punto,Fijo
2,OrdenadorTamano,orden,Micro | Pequeña | Mediana | Grande | NoClasifi...,Fijo
3,División,grupo,NroLicitacion,Fijo
4,División,semilla,42,Fijo
5,División,fracción de prueba por grupos,0.2,Fijo


## 6. Qué llevar al informe técnico

- **Diseño:** enlazar `src/pipeline_licitaciones.py` y explicar qué hace cada clase y cómo se ejecuta el pipeline. Usar la bitácora como evidencia del patrón Observer.
- **Equivalencia:** incluir el resultado de `assert_frame_equal` y aclarar que los controles de Fase 2, como los atípicos, no se usaron para eliminar filas.
- **Eficiencia:** incorporar la tabla de tiempos, el tamaño del conjunto y la explicación de por qué ambas implementaciones tienen complejidad lineal.
- **Trazabilidad:** enlazar el commit que incorpore el notebook y el módulo. Ejecutar todas las celdas en el entorno del repositorio antes de la entrega.